# Lista de Exercícios 2.2 — Comparação de Métodos de Localização de Raízes

**Atividade:** Replicar a comparação da questão 4 da lista de exercícios, incluindo agora os métodos de **Newton-Raphson** e **Secante**.

Serão comparados:
- Bisseção
- Falsa Posição
- Iteração de Ponto Fixo
- Newton-Raphson
- Secante

A comparação considera raiz encontrada, convergência, número de iterações e comportamento de cada método.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.optimize as opt
import time

TOL = 1e-10
MAX_ITER = 200

## Funções auxiliares

A Bisseção, o Ponto Fixo, Newton-Raphson e Secante são executados por `scipy.optimize`, seguindo a abordagem do material.

Para o Ponto Fixo usamos `method="iteration"`, pois queremos a iteração simples. A Falsa Posição é implementada manualmente, já que ela não é um método disponível no `root_scalar` utilizado aqui.

In [ ]:
def falsa_posicao(f, a, b, tol=TOL, max_iter=MAX_ITER):
    fa, fb = f(a), f(b)
    if fa * fb >= 0:
        return np.nan, 0, False

    for i in range(1, max_iter + 1):
        x = b - fb * (a - b) / (fa - fb)
        fx = f(x)

        if abs(fx) <= tol:
            return x, i, True

        if fa * fx < 0:
            b, fb = x, fx
        else:
            a, fa = x, fx

    return x, max_iter, False


def bissecao(f, a, b):
    try:
        r = opt.root_scalar(f, method="bisect", bracket=[a, b],
                            xtol=TOL, maxiter=MAX_ITER)
        return r.root, r.iterations, r.converged
    except Exception:
        return np.nan, MAX_ITER, False


def ponto_fixo(g, x0):
    try:
        r = opt.fixed_point(g, x0, xtol=TOL,
                            maxiter=MAX_ITER, method="iteration")
        return float(r), np.nan, True
    except Exception:
        return np.nan, MAX_ITER, False


def newton(f, df, x0):
    try:
        r = opt.root_scalar(f, fprime=df, x0=x0,
                            method="newton", xtol=TOL, maxiter=MAX_ITER)
        return r.root, r.iterations, r.converged
    except Exception:
        return np.nan, MAX_ITER, False


def secante(f, x0, x1):
    try:
        r = opt.root_scalar(f, x0=x0, x1=x1,
                            method="secant", xtol=TOL, maxiter=MAX_ITER)
        return r.root, r.iterations, r.converged
    except Exception:
        return np.nan, MAX_ITER, False


def executar(nome, metodo, *args):
    inicio = time.perf_counter()
    raiz, it, conv = metodo(*args)
    tempo = time.perf_counter() - inicio

    return {
        "Método": nome,
        "Raiz": raiz,
        "Iterações": it,
        "Convergiu": conv,
        "Tempo (s)": tempo
    }


def comparar(f, df, g, a, b, x0, x1, titulo):
    resultados = [
        executar("Bisseção", bissecao, f, a, b),
        executar("Falsa Posição", falsa_posicao, f, a, b),
        executar("Ponto Fixo", ponto_fixo, g, x0),
        executar("Newton-Raphson", newton, f, df, x0),
        executar("Secante", secante, f, x0, x1)
    ]

    tabela = pd.DataFrame(resultados)
    print(titulo)
    display(tabela)
    return tabela

# Questão 4

## a) $f_1(x)=2x^4+4x^3+3x^2-10x-15$, $x^*\in[0,3]$

Para o Ponto Fixo:

$$g(x)=\frac{2x^4+4x^3+3x^2-15}{10}.$$

Para Newton-Raphson:

$$f'_1(x)=8x^3+12x^2+6x-10.$$

In [ ]:
f1 = lambda x: 2*x**4 + 4*x**3 + 3*x**2 - 10*x - 15
df1 = lambda x: 8*x**3 + 12*x**2 + 6*x - 10
g1 = lambda x: (2*x**4 + 4*x**3 + 3*x**2 - 15) / 10

x = np.linspace(0, 3, 500)
plt.plot(x, f1(x))
plt.axhline(0, linewidth=0.8)
plt.title(r"$f_1(x)=2x^4+4x^3+3x^2-10x-15$")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.grid()
plt.show()

tabela_f1 = comparar(f1, df1, g1, 0, 3, 3, 2.9, "Comparação — f1")

## b) $f_2(x)=(x+3)(x+1)(x-2)^3$, $x^*\in[0,5]$

Para o Ponto Fixo, adotamos uma transformação $g(x)=x-\alpha f(x)$, com $\alpha=10^{-4}$.

Essa escolha ilustra uma característica importante do Ponto Fixo: a forma de $g(x)$ influencia diretamente a convergência.

In [ ]:
f2 = lambda x: (x + 3)*(x + 1)*(x - 2)**3
df2 = lambda x: 3*(x + 3)*(x + 1)*(x - 2)**2 + 2*(x + 3)*(x - 2)**3
alpha2 = 1e-4
g2 = lambda x: x - alpha2*f2(x)

x = np.linspace(0, 5, 500)
plt.plot(x, f2(x))
plt.axhline(0, linewidth=0.8)
plt.title(r"$f_2(x)=(x+3)(x+1)(x-2)^3$")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.grid()
plt.show()

tabela_f2 = comparar(f2, df2, g2, 0, 5, 5, 4.9, "Comparação — f2")

## c) $f_3(x)=5x^3+x^2-e^{1-2x}+\cos(x)+20$, $x^*\in[-5,5]$

A derivada utilizada no Newton-Raphson é:

$$f'_3(x)=15x^2+2x+2e^{1-2x}-\sin(x).$$

No Ponto Fixo:

$$g(x)=\sqrt[3]{\frac{-x^2+e^{1-2x}-\cos(x)-20}{5}}.$$

In [ ]:
f3 = lambda x: 5*x**3 + x**2 - np.exp(1 - 2*x) + np.cos(x) + 20
df3 = lambda x: 15*x**2 + 2*x + 2*np.exp(1 - 2*x) - np.sin(x)
g3 = lambda x: np.cbrt((-x**2 + np.exp(1 - 2*x) - np.cos(x) - 20) / 5)

x = np.linspace(-5, 5, 500)
plt.plot(x, f3(x))
plt.axhline(0, linewidth=0.8)
plt.title(r"$f_3(x)=5x^3+x^2-e^{1-2x}+\cos(x)+20$")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.grid()
plt.show()

tabela_f3 = comparar(f3, df3, g3, -5, 5, -5, -4.9, "Comparação — f3")

## d) $f_4(x)=x\sin(x)+4$, $x^*\in[1,5]$

A derivada é:

$$f'_4(x)=\sin(x)+x\cos(x).$$

Para o Ponto Fixo, usamos $g(x)=x+0.1f(x)$.

In [ ]:
f4 = lambda x: x*np.sin(x) + 4
df4 = lambda x: np.sin(x) + x*np.cos(x)
g4 = lambda x: x + 0.1*f4(x)

x = np.linspace(1, 5, 500)
plt.plot(x, f4(x))
plt.axhline(0, linewidth=0.8)
plt.title(r"$f_4(x)=x\sin(x)+4$")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.grid()
plt.show()

tabela_f4 = comparar(f4, df4, g4, 1, 5, 1, 1.1, "Comparação — f4")

## e) $f_5(x)=(x-3)^5\ln(x)$, $x^*\in[2,5]$

A derivada é:

$$f'_5(x)=5(x-3)^4\ln(x)+\frac{(x-3)^5}{x}.$$

Para o Ponto Fixo usamos $g(x)=x-0.001f(x)$.

In [ ]:
f5 = lambda x: (x - 3)**5 * np.log(x)
df5 = lambda x: 5*(x - 3)**4*np.log(x) + (x - 3)**5/x
g5 = lambda x: x - 0.001*f5(x)

x = np.linspace(2, 5, 500)
plt.plot(x, f5(x))
plt.axhline(0, linewidth=0.8)
plt.title(r"$f_5(x)=(x-3)^5\ln(x)$")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.grid()
plt.show()

tabela_f5 = comparar(f5, df5, g5, 2, 5, 5, 4.9, "Comparação — f5")

## f) $f_6(x)=x^{10}-1$, $x^*\in[0.8,1.2]$

A derivada é:

$$f'_6(x)=10x^9.$$

Para o Ponto Fixo usamos $g(x)=x-0.05f(x)$.

In [ ]:
f6 = lambda x: x**10 - 1
df6 = lambda x: 10*x**9
g6 = lambda x: x - 0.05*f6(x)

x = np.linspace(0.8, 1.2, 500)
plt.plot(x, f6(x))
plt.axhline(0, linewidth=0.8)
plt.title(r"$f_6(x)=x^{10}-1$")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.grid()
plt.show()

tabela_f6 = comparar(f6, df6, g6, 0.8, 1.2, 1.2, 0.8, "Comparação — f6")

# Tabela de comparação geral

In [ ]:
tabelas = {
    "f1": tabela_f1,
    "f2": tabela_f2,
    "f3": tabela_f3,
    "f4": tabela_f4,
    "f5": tabela_f5,
    "f6": tabela_f6
}

linhas = []
for funcao, tabela in tabelas.items():
    for _, row in tabela.iterrows():
        linhas.append({
            "Função": funcao,
            "Método": row["Método"],
            "Raiz": row["Raiz"],
            "Iterações": row["Iterações"],
            "Convergiu": row["Convergiu"],
            "Tempo (s)": row["Tempo (s)"]
        })

tabela_final = pd.DataFrame(linhas)
display(tabela_final)

# Análise dos resultados

### Bisseção
A Bisseção mantém um intervalo contendo uma mudança de sinal e o reduz progressivamente. Por isso, é bastante robusta e previsível, mas tende a exigir mais iterações.

### Falsa Posição
A Falsa Posição também mantém o intervalo, mas usa a interseção da reta com o eixo $x$ para produzir a próxima aproximação. Pode superar a Bisseção em alguns problemas, mas pode ficar lenta quando uma das extremidades permanece praticamente fixa.

### Ponto Fixo
O Ponto Fixo é o método mais dependente da formulação. Não basta conhecer $f(x)$: é preciso escolher uma função $g(x)$ adequada. Uma escolha ruim pode causar convergência lenta, oscilação ou divergência. Por isso, resultados muito diferentes dos demais métodos são esperados em alguns itens.

### Newton-Raphson
Newton-Raphson utiliza a derivada:

$$x_{n+1}=x_n-\frac{f(x_n)}{f'(x_n)}.$$

Quando o ponto inicial está em uma região favorável, sua convergência costuma ser muito rápida. Em contrapartida, é mais sensível ao ponto inicial e a regiões onde a derivada é pequena ou muda de comportamento.

### Secante
A Secante aproxima a derivada numericamente usando dois pontos:

$$
x_{n+1}=x_n-f(x_n)\frac{x_n-x_{n-1}}{f(x_n)-f(x_{n-1})}.
$$

Ela tem a vantagem de não exigir $f'(x)$, podendo ser rápida como Newton-Raphson em muitos casos. Entretanto, também é um método aberto e, portanto, menos previsível que os métodos intervalares.

### Conclusão
As diferenças observadas são consequência das características matemáticas dos métodos. Bisseção e Falsa Posição privilegiam a manutenção de um intervalo com mudança de sinal; Ponto Fixo depende fortemente de $g(x)$; Newton-Raphson depende da derivada e do ponto inicial; e Secante procura obter comportamento rápido sem calcular explicitamente a derivada.

Assim, **não há um método universalmente melhor**. Em problemas em que a robustez é prioridade, métodos intervalares tendem a ser mais seguros. Quando boas aproximações iniciais estão disponíveis, Newton-Raphson e Secante podem reduzir bastante o número de iterações.